# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [36]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

if api_key and api_key.startswith('gsk_') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'openai/gpt-oss-20b'
openai = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=api_key)

API key looks good so far


In [37]:
links = fetch_website_links("https://huggingface.co")
links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/storage',
 '/docs',
 '/enterprise',
 '/pricing',
 '/tasks',
 '/chat',
 '/collections',
 '/languages',
 '/organizations',
 '/blog/zh',
 '/posts',
 '/papers',
 '/hardware',
 '/learn',
 '/join/discord',
 'https://discuss.huggingface.co/',
 'https://github.com/huggingface',
 '/enterprise',
 '/pro',
 '/support',
 '/inference/models',
 '/inference-endpoints',
 '/storage',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/deepseek-ai/DeepSeek-V4.1-Flash',
 '/Edge0/Edge0-35B-A3B-preview',
 '/openbmb/MiniCPM5-2B',
 '/nex-agi/Nex-N2.5-mini',
 '/Qwen/Qwen3.8-27B',
 '/models',
 '/spaces/kulkas2pintu/wan555',
 '/spaces/ravenrose996688/Krea-2-Turbo_I2I',
 '/spaces/Pepe104/MiniMax-H3-Turbo-Lora-UNCENSORED',
 '/spaces/kulkas2pintu/QWEN_EDIT_IMAGE',
 '/spaces/Cockdaddyfuck/Wan_2.2_I2V_14B_Custom_Lora_Wow',
 '/spaces',
 '/datasets/nyu-mll/glue',
 '/datasets/stanfordnlp/imdb',
 '/datasets/rajpurkar/squad',
 '/datasets/openbmb/UltraData-SFT-Agent-2609',
 '/data

## First step: Have nvidia/nemotron-3.5-lightning:free figure out which links are relevant

### Use a call to nvidia/nemotron-3.5-lightning:free to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [38]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [39]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [40]:
print(get_links_user_prompt("https://huggingface.co"))


Here is the list of links on the website https://huggingface.co -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/tasks
/chat
/collections
/languages
/organizations
/blog/zh
/posts
/papers
/hardware
/learn
/join/discord
https://discuss.huggingface.co/
https://github.com/huggingface
/enterprise
/pro
/support
/inference/models
/inference-endpoints
/storage
/login
/join
/spaces
/models
/deepseek-ai/DeepSeek-V4.1-Flash
/Edge0/Edge0-35B-A3B-preview
/openbmb/MiniCPM5-2B
/nex-agi/Nex-N2.5-mini
/Qwen/Qwen3.8-27B
/models
/spaces/kulkas2pintu/wan555
/spaces/ravenrose996688/Krea-2-Turbo_I2I
/spaces/Pepe104/MiniMax-H3-Turbo-Lora-UNCENSORED
/spaces/kulkas2pintu/QWEN_EDIT_IMAGE
/spaces/Cockdaddyfuck/Wan_2.2_I2V_14B_Custom_Lora_Wow
/spaces
/datasets/nyu-mll

In [41]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [42]:
print(select_relevant_links("https://huggingface.co"))

{'links': [{'type': 'company page', 'url': 'https://huggingface.co'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'learn page', 'url': 'https://huggingface.co/learn'}, {'type': 'documentation page', 'url': 'https://huggingface.co/docs'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'github page', 'url': 'https://github.com/huggingface'}, {'type': 'twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'linkedin page', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'discord community', 'url': 'https://huggingface.co/join/discord'}, {'type': 'discussions forum', 'url': 'https://discuss.huggingface.co'}]}


In [43]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [44]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling openai/gpt-oss-20b
Found 7 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'proficiency page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [45]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-20b
Found 11 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'chat page', 'url': 'https://huggingface.co/chat'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'docs page', 'url': 'https://huggingface.co/docs'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'support page', 'url': 'https://huggingface.co/support'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [46]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [47]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-20b
Found 4 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4.1-Flash
Updated
4 days ago
•
288k
•
2.37k
Edge0/Edge0-35B-A3B-preview
Updated
about 10 hours ago
•
8.11k
•
1.59k
openbmb/MiniCPM5-2B
Updated
2 days ago
•
207k
•
1.37k
nex-agi/Nex-N2.5-mini
Updated
6 days ago
•
4.54k
•
776
Qwen/Qwen3.8-27B
Updated
about 1 month ago
•
7.7M
•
15.1k
Browse 2M+ models
Spaces
Running
on
Zero
MCP
2.11k
Wan2.2 14B Fast Preview
🐌
2.

In [48]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [49]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [50]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-20b
Found 11 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4.1-Flash\nUpdated\n4 days ago\n•\n288k\n•\n2.37k\nEdge0/Edge0-35B-A3B-preview\nUpdated\nabout

In [57]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [58]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-20b
Found 15 relevant links


# Hugging Face  
**The AI community building the future**

---

## About Us  
Hugging Face is a global hub where researchers, developers, and companies come together to create, discover, and collaborate on machine‑learning models, datasets, and applications. With a growing catalog of **2 million+ models** and **500 k+ datasets**, we make cutting‑edge AI accessible to anyone—from hobbyists to Fortune 500 enterprises.

* Open‑source first: our entire stack is free to use, fork, and improve.
* Community‑driven: thousands of contributors help shape the future of AI every day.
* Inclusive culture: we celebrate diversity of thought, background, and skill level.

---

## Core Offerings  

| Feature | What it gives you |
|---------|-------------------|
| **Model Hub** | Browse, test, and download the world’s largest collection of ML models, from language and vision to audio and reinforcement learning. |
| **Datasets** | 500 k+ public datasets ready to load into your training pipeline with a single line of code. |
| **Spaces** | Deploy end‑to‑end ML apps in seconds. No infra worries—run on our Zero‑MCP or Zero‑Agents hosting. |
| **Inference Endpoints** | Production‑ready, auto‑scaling API endpoints for any model, from research prototypes to commercial workloads. |
| **Storage Buckets** | Secure, globally‑distributed object storage tailored for ML data, models, and logs. |
| **HuggingChat** | A conversational AI framework you can train, fine‑tune, and deploy for customer support, virtual assistants, and more. |
| **Enterprise Support & PRO** | Dedicated help desks, SLA guarantees, and advanced tooling for large teams and mission‑critical applications. |

---

## Why Customers Trust Us  

| Customer Type | How we help |
|---------------|-------------|
| **Startups & SMEs** | Rapid prototyping with pre‑built models, low‑cost inference, and community mentorship. |
| **Large Enterprises** | Scalable inference, enterprise‑grade support, and integration into existing ML pipelines. |
| **Academic & Research Labs** | Open‑source code, reproducible datasets, and a collaborative platform for peer review. |
| **Developers & Hobbyists** | Zero‑cost hosting for demos, intuitive web UI, and a vibrant community on Discord and GitHub. |

---

## Join the Hugging Face Community  

- **Discord** – real‑time chat, AMA sessions, and code reviews.  
- **Forum** – deep dives into model architecture, training tricks, and best practices.  
- **GitHub** – contribute to the open‑source stack, submit issues, and propose new features.  
- **Blog & Daily Papers** – stay up‑to‑date with the latest research and industry trends.

---

## Careers at Hugging Face  

We’re building the next generation of AI tools. If you’re passionate about open science, scalable ML, and a collaborative culture, consider joining us:

- **Machine Learning Engineers** – train, fine‑tune, and deploy state‑of‑the‑art models.  
- **Software Engineers** – build the infrastructure that powers 2 million+ models.  
- **Product & Community Managers** – shape the user experience and drive community growth.  
- **Data Scientists & Research Fellows** – push the boundaries of language, vision, and multimodal AI.  

*We value curiosity, inclusivity, and the desire to learn and share. Explore open positions at [careers.huggingface.co](https://huggingface.co/careers).*

---

## Get Started  

1. **Explore** – Browse 2 million+ models and 500 k+ datasets.  
2. **Try** – Run an app on Spaces with zero infrastructure.  
3. **Build** – Deploy a production‑ready endpoint with our inference service.  
4. **Collaborate** – Contribute back to the community and help shape the future of AI.

> “Hugging Face is the single most important open‑source community for AI today.” – *Industry Insider*

---

**Contact**  
Website: [huggingface.co](https://huggingface.co)  
Email: careers@huggingface.co  
Social: @huggingface (Twitter) | #huggingface (Discord)  

*Together, we’re not just building models – we’re building the future of AI.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [63]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [64]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-20b
Found 2 relevant links


# Hugging Face – The AI Community Building the Future

---

## Who We Are  
Hugging Face is the global hub where researchers, developers, and businesses share, build, and deploy machine‑learning models, datasets, and applications.  
- **Mission** – Accelerate the creation and adoption of AI by fostering open collaboration.  
- **Community** – 2 M+ models, 1 M+ applications, 500 k+ datasets and a vibrant forum, Discord, and GitHub presence.

---

## Core Offerings

### Models  
- **Public Model Hub** – Browse and use over **2 million** ready‑to‑go models, from language models like Qwen 3.8‑27B to multimodal tools such as MiniCPM5‑2B.  
- **Zero‑Shot & Fine‑Tuning** – Leverage HF’s open‑source stack for quick experimentation and production deployment.

### Datasets  
- **Open‑Source Catalog** – Explore 500 k+ datasets (e.g., GLUE, IMDb, SQuAD) with community‑curated metadata and usage statistics.

### Spaces  
- **Run & Share Apps** – Deploy interactive ML apps with the “Spaces” platform; 1 M+ applications including image‑to‑video, text‑to‑image, and video generation demos.

### Buckets & Inference  
- **Storage & Compute** – Unified storage buckets for models and data.  
- **Inference Providers & Endpoints** – Seamless deployment of models across cloud providers or on‑prem.

### Enterprise Solutions  
- **Hugging Face PRO** – Enterprise support, advanced security, and compliance.  
- **Enterprise Pricing** – Flexible plans for teams, from startups to large corporations.

---

## Culture & Community  
- **Open‑Source First** – All core libraries (Transformers, Datasets, Diffusers) are MIT‑licensed and hosted on GitHub.  
- **Inclusive Collaboration** – Regular workshops, tutorials, and a global Discord community where anyone can contribute.  
- **Learning & Resources** – Documentation, “Learn” tutorials, and a “Daily Papers” feed keep users at the cutting edge.

---

## For Customers  
- **Fast Time‑to‑Value** – Instantly load pre‑trained models, fine‑tune on custom data, and expose them via APIs.  
- **Scalable Deployment** – From local inference to cloud‑scale with zero‑code migrations.  
- **Community Insights** – Tap into the collective wisdom of thousands of contributors for best‑practice guidance.

---

## For Investors  
- **High Growth** – Rapid adoption in academia, industry, and the open‑source ecosystem.  
- **Strategic Partnerships** – Collaborations with major cloud providers and hardware vendors.  
- **Monetization Channels** – Enterprise subscriptions, managed inference, and professional services.

---

## Careers & Talent  
- **Open Positions** – Hugging Face welcomes data scientists, ML engineers, product managers, and community builders.  
- **Impactful Work** – Join a team that shapes the future of AI and works on projects that touch millions of developers worldwide.  
- **Diverse & Inclusive** – A culture that values collaboration, curiosity, and continuous learning.

> *Visit the careers page for the latest opportunities and learn how you can contribute to the AI community.*

---

## Get Involved  
- **Explore** – Visit the [Hub](https://huggingface.co) to browse models, datasets, and Spaces.  
- **Learn** – Dive into tutorials and the “Learn” section.  
- **Contribute** – Fork, star, or submit a pull request on GitHub.  
- **Connect** – Join the Discord or forum for real‑time discussions.

---

*Hugging Face – Empowering the world to build, share, and deploy AI together.*

In [65]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-20b
Found 3 relevant links


# Hugging Face – The AI Community Building the Future

---

## Who We Are  
Hugging Face is the **collaboration platform for the machine learning community**.  
- **Open‑source first**: Every model, dataset, and application lives on the Hugging Face Hub, accessible to anyone.  
- **Community‑driven**: Thousands of researchers, engineers, and hobbyists contribute, review, and build together.  
- **Fast‑track innovation**: With over 2 million models and 500 k+ datasets, you can start experimenting in minutes, not months.

---

## What We Offer

| Feature | What it means | Why it matters |
|---------|---------------|----------------|
| **Models** | 2 M+ pre‑trained models, from transformers to vision & speech | Reduce 90 % of model development time |
| **Datasets** | 500 k+ curated datasets, open‑source & community‑maintained | Reliable data without the licensing headaches |
| **Spaces** | Run ML demos and apps with a single click | Share prototypes instantly with zero deployment |
| **Buckets** | Scalable, private storage for datasets & training assets | Seamless data pipelines for production |
| **Enterprise** | Hugging Face PRO, inference endpoints, support contracts | Managed AI at scale with SLAs and dedicated assistance |
| **Docs & Learning** | Extensive tutorials, API docs, and community forums | Lower the barrier to entry for new talent |

---

## Our Culture

- **Collaboration over competition** – Contributions are reviewed by peers and merged into the community repo.  
- **Transparency** – All code, data, and model weights are openly licensed.  
- **Learning‑first** – Regular blog posts, papers, and “State of Open‑Source AI” reports keep the community informed.  
- **Diverse voices** – A vibrant Discord, forum, and global mailing list where every perspective is valued.

---

## Who Uses Hugging Face

| Segment | Examples |
|---------|----------|
| **Start‑ups** | Quickly prototype NLP or CV solutions with Hugging Face models. |
| **Enterprise** | Deploy inference endpoints in production, backed by Hugging Face PRO support. |
| **Researchers** | Publish models and datasets, benchmark new algorithms. |
| **Developers** | Build AI‑powered apps via Spaces and the Hub API. |
| **Education** | Use the open‑source stack to teach ML in classrooms worldwide. |

---

## Careers & Opportunities

- **Team size**: 185+ members worldwide (data scientists, engineers, community managers, product, and support).  
- **Growth mindset**: Frequent hack‑days, research talks, and open‑source contributions are part of the daily routine.  
- **Where to apply**: Visit the official **Careers** page on the website to see current openings in engineering, product, data science, and community roles.  
- **Why work here**: Freedom to experiment, impact real‑world AI, and collaborate with a global network of innovators.

---

## Get Involved

1. **Explore the Hub** – Browse models, datasets, or Spaces.  
2. **Contribute** – Fork, improve, and submit pull requests.  
3. **Join the community** – Discord, forum, or GitHub discussions.  
4. **Share your work** – Publish articles or blog posts on the Hugging Face blog.  

**Brand assets** (logos, colors, style guidelines) are freely downloadable from the Brand Assets page to help you showcase your projects.

---

> *“Hugging Face is the collaboration platform for the machine learning community.”* – Hugging Face

Discover the next breakthrough together—**join Hugging Face**.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>